# Olist Brazilian E-Commerce Business Analysis
## Phase 5: Feature Engineering Pipeline

This notebook focuses on engineering critical features for logistics, loyalty, and customer analytics, culminating in a single unified **Master Orders Dataset** containing all order-level properties, customer parameters, payment metrics, and review attributes. We use the modular calculations defined in `src/features.py`.

In [1]:
import os
import sys
import pandas as pd
import numpy as np

# Set path to import our modules
sys.path.append(os.path.abspath("../"))
from src.features import build_master_features

PROCESSED_DIR = "../data/processed"

### 1. Load Processed Datasets
Let's load the cleaned tables from `data/processed/` generated during Phase 4.

In [2]:
customers = pd.read_csv(os.path.join(PROCESSED_DIR, "olist_customers_dataset.csv"))
geolocation = pd.read_csv(os.path.join(PROCESSED_DIR, "olist_geolocation_dataset.csv"))
order_items = pd.read_csv(os.path.join(PROCESSED_DIR, "olist_order_items_dataset.csv"))
order_payments = pd.read_csv(os.path.join(PROCESSED_DIR, "olist_order_payments_dataset.csv"))
order_reviews = pd.read_csv(os.path.join(PROCESSED_DIR, "olist_order_reviews_dataset.csv"))
orders = pd.read_csv(os.path.join(PROCESSED_DIR, "olist_orders_dataset.csv"))
products = pd.read_csv(os.path.join(PROCESSED_DIR, "olist_products_dataset.csv"))
sellers = pd.read_csv(os.path.join(PROCESSED_DIR, "olist_sellers_dataset.csv"))

### 2. Generate Master Orders Table & Features
Let's execute the feature extraction pipeline by passing the cleaned tables into `build_master_features`.

In [3]:
master_orders = build_master_features(
    orders_df=orders,
    customers_df=customers,
    items_df=order_items,
    payments_df=order_payments,
    reviews_df=order_reviews,
    products_df=products,
    sellers_df=sellers
)

print(f"Master Orders Dataset Shape: {master_orders.shape}")
print("\n--- First 3 Rows Preview ---")
display(master_orders.head(3))

Master Orders Dataset Shape: (99441, 45)

--- First 3 Rows Preview ---


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,shipping_duration,late_delivery_flag,customer_total_orders,customer_total_spend,first_purchase_date,customer_lifetime,repeat_customer_flag,high_value_customer,revenue_category,seller_state_match
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,6.062650,0,2,82.82,2017-09-04 11:26:38,27.979109,1,0,low,1
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,12.039410,0,1,141.46,2018-07-24 20:41:37,0.000000,0,0,medium,0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,9.178113,0,1,179.12,2018-08-08 08:38:49,0.000000,0,0,medium,0


### 3. Review Generated Features & Distributions
Let's check the values of some newly generated columns.

In [4]:
feature_cols = [
    'order_size', 'revenue_per_order', 'payment_installments',
    'payment_category', 'avg_review_score', 'review_category',
    'delivery_days', 'delivery_delay', 'shipping_duration',
    'late_delivery_flag', 'customer_lifetime', 'repeat_customer_flag',
    'high_value_customer', 'revenue_category', 'seller_state_match'
]
display(master_orders[feature_cols].describe(include='all'))

,order_size,revenue_per_order,payment_installments,payment_category,avg_review_score,review_category,delivery_days,delivery_delay,shipping_duration,late_delivery_flag,customer_lifetime,repeat_customer_flag,high_value_customer,revenue_category,seller_state_match
count,99441.000000,99441.000000,99441.000000,99441,99441.000000,99441,96476.000000,96476.000000,96475.000000,99441.000000,99441.000000,99441.000000,99441.000000,99441,99441.000000
unique,NaN,NaN,NaN,6,NaN,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN
top,NaN,NaN,NaN,credit_card,NaN,positive,NaN,NaN,NaN,NaN,NaN,NaN,NaN,medium,NaN
freq,NaN,NaN,NaN,76132,NaN,76028,NaN,NaN,NaN,NaN,NaN,NaN,NaN,61766,NaN
mean,1.132833,159.326166,2.930502,NaN,4.055230,NaN,12.558702,-11.179120,9.330547,0.078710,2.933987,0.063777,0.100009,NaN,0.356784
std,0.545666,220.058804,2.715678,NaN,1.387967,NaN,9.546530,10.186113,8.760122,0.269287,26.256760,0.244356,0.300014,NaN,0.479053
min,0.000000,0.000000,0.000000,NaN,0.000000,NaN,0.533414,-146.016123,-16.096169,0.000000,0.000000,0.000000,0.000000,NaN,0.000000
25%,1.000000,61.050000,1.000000,NaN,4.000000,NaN,6.766403,-16.244384,4.099948,0.000000,0.000000,0.000000,0.000000,NaN,0.000000
50%,1.000000,104.560000,2.000000,NaN,5.000000,NaN,10.217755,-11.948941,7.099769,0.000000,0.000000,0.000000,0.000000,NaN,0.000000
75%,1.000000,176.000000,4.000000,NaN,5.000000,NaN,15.720327,-6.390000,12.029115,0.000000,0.000000,0.000000,0.000000,NaN,1.000000


#### 3.1 Repeat Customers Analysis
Let's evaluate the proportion of repeat buyers vs one-time buyers.

In [5]:
print("Repeat Customer Distribution:")
print(master_orders['repeat_customer_flag'].value_counts(normalize=True) * 100)

Repeat Customer Distribution:
repeat_customer_flag
0    93.622349
1     6.377651
Name: proportion, dtype: float64


#### 3.2 Late Delivery Flag
Let's check the percentage of orders delivered after the estimated delivery date.

In [6]:
print("Late Delivery Rate (LDR):")
print(master_orders['late_delivery_flag'].value_counts(normalize=True) * 100)

Late Delivery Rate (LDR):
late_delivery_flag
0    92.129001
1     7.870999
Name: proportion, dtype: float64


#### 3.3 High-Value Customers
Let's see what threshold spending splits the top 10% from the remaining 90%.

In [7]:
customer_rev = master_orders.groupby('customer_unique_id')['revenue_per_order'].sum()
print(f"90th Percentile Spending Threshold: {customer_rev.quantile(0.90):.2f} BRL")
print(f"Max Customer Spending: {customer_rev.max():.2f} BRL")

90th Percentile Spending Threshold: 318.34 BRL
Max Customer Spending: 13664.08 BRL


### 4. Export Master Orders Dataset
Let's save the engineered dataset to `data/processed/` for use in visual profiling and analysis phases.

In [8]:
master_orders.to_csv(os.path.join(PROCESSED_DIR, "master_orders_dataset.csv"), index=False)
print("Master Orders Dataset successfully exported to data/processed!")

Master Orders Dataset successfully exported to data/processed!
